# P02 第一部分：数据获取

本 Notebook 完成四类数据的下载：
1. 10 只 A 股股票的日度后复权行情
2. 沪深 300 + 创业板指
3. CPI 同比增速 + 人民币/美元汇率
4. 财务指标（ROE、销售毛利率）

In [1]:
from pathlib import Path
from datetime import datetime
import os
import time

import numpy as np
import pandas as pd

# ---------- 项目根目录 ----------
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    ROOT = Path(r"G:/ganlijie/ds_homework/dshw-p01")

START_DATE = "20200101"
END_DATE = datetime.now().strftime("%Y%m%d")

STOCK_LIST = [
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "601633", "name": "长城汽车", "industry": "汽车"},
    {"code": "000002", "name": "万科A", "industry": "房地产"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "name": "五粮液", "industry": "白酒"},
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    {"code": "600941", "name": "中国移动", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

DIRS = {
    "stock": ROOT / "data" / "stock",
    "index": ROOT / "data" / "index",
    "macro": ROOT / "data" / "macro",
    "finance": ROOT / "data" / "finance",
    "clean": ROOT / "data" / "clean",
    "combined": ROOT / "data" / "combined",
    "output": ROOT / "output",
}
LOG_FILE = ROOT / "download_log.txt"
COMBINED_CSV = DIRS["combined"] / "combined_data.csv"
STOCK_CLEAN_CSV = DIRS["clean"] / "stock_clean.csv"
STOCK_CLEAN_PARQUET = DIRS["clean"] / "stock_clean.parquet"
FIN_DB = DIRS["combined"] / "fin_data.db"
RF_DAILY = 0.02 / 252
TRADING_DAYS = 252
FINANCE_INDICATORS = ["净资产收益率(ROE)", "销售净利率"]

def setup_directories():
    for p in DIRS.values():
        p.mkdir(parents=True, exist_ok=True)

def log_download(status, tag, detail=""):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {status:<7} {tag}"
    if detail:
        line += f"  {detail}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def stock_csv_path(code):
    return DIRS["stock"] / f"stock_{code}.csv"

def index_csv_path(code):
    return DIRS["index"] / f"index_{code}.csv"

setup_directories()

try:
    from IPython.display import display
except ImportError:
    display = print

print("项目根目录:", ROOT.resolve())

# 初始化下载日志
if LOG_FILE.exists():
    LOG_FILE.write_text('', encoding='utf-8')
print('待下载:', [s['name'] for s in STOCK_LIST])

项目根目录: E:\ds_homework\T02\ds_homework\ds_homework\dshw-p01
待下载: ['工商银行', '招商银行', '比亚迪', '长城汽车', '万科A', '贵州茅台', '五粮液', '中国石油', '中国移动', '顺丰控股']


In [2]:
import akshare as ak

STOCK_COL_MAP = {
    "日期": "date", "开盘": "open", "收盘": "close", "最高": "high",
    "最低": "low", "成交量": "volume", "成交额": "amount",
}

def rename_stock_cols(df):
    out = df.rename(columns=STOCK_COL_MAP)
    return out[["date", "open", "close", "high", "low", "volume", "amount"]]

def download_all_stocks():
    results = {}
    for item in STOCK_LIST:
        code = item["code"]
        tag = f"stock_{code}"
        try:
            raw = ak.stock_zh_a_hist(
                symbol=code, period="daily",
                start_date=START_DATE, end_date=END_DATE, adjust="hfq",
            )
            if raw is None or raw.empty:
                raise ValueError("No data returned")
            df = rename_stock_cols(raw)
            df["code"] = str(code).zfill(6)
            df.to_csv(stock_csv_path(code), index=False, encoding="utf-8-sig")
            results[code] = df
            log_download("SUCCESS", tag, f"shape={df.shape}")
            time.sleep(0.3)
        except Exception as e:
            log_download("FAILED", tag, f"Error: {e}")
    return results

def download_all_indices():
    sina_map = {"000300": "sh000300", "399006": "sz399006"}
    results = {}
    for code, sym in sina_map.items():
        tag = f"index_{code}"
        try:
            raw = ak.stock_zh_index_daily(symbol=sym)
            raw["date"] = pd.to_datetime(raw["date"])
            mask = (raw["date"] >= "2020-01-01") & (raw["date"] <= pd.Timestamp.today())
            df = raw.loc[mask, ["date", "open", "close", "high", "low", "volume"]].copy()
            df["amount"] = pd.NA
            df.to_csv(index_csv_path(code), index=False, encoding="utf-8-sig")
            results[code] = df
            log_download("SUCCESS", tag, f"shape={df.shape}")
            time.sleep(0.3)
        except Exception as e:
            log_download("FAILED", tag, f"Error: {e}")
    return results

def parse_cn_month(s):
    s = str(s).replace("月份", "").replace("月", "")
    year, month = s.split("年")
    return pd.Timestamp(year=int(year), month=int(month), day=1)

def download_macro_cpi():
    tag = "macro_cpi"
    try:
        raw = ak.macro_china_cpi_monthly()
        df = raw.rename(columns={"日期": "date", "今值": "value"})[["date", "value"]]
        df["date"] = pd.to_datetime(df["date"])
        df = df.dropna(subset=["value"]).sort_values("date")
        df = df[df["date"] >= "2020-01-01"]
        df["indicator"] = "cpi"
        df.to_csv(DIRS["macro"] / "macro_cpi.csv", index=False, encoding="utf-8-sig")
        log_download("SUCCESS", tag, f"shape={df.shape}")
        return df
    except Exception as e:
        log_download("FAILED", tag, f"Error: {e}")
        return pd.DataFrame()

def download_macro_m2():
    tag = "macro_m2"
    try:
        raw = ak.macro_china_money_supply()
        col_val = [c for c in raw.columns if "M2" in c and "同比" in c][0]
        df = raw.rename(columns={"月份": "date_raw", col_val: "value"})
        df["date"] = df["date_raw"].map(parse_cn_month)
        df = df[["date", "value"]].dropna()
        df = df[df["date"] >= "2020-01-01"].sort_values("date")
        df["indicator"] = "m2"
        df.to_csv(DIRS["macro"] / "macro_m2.csv", index=False, encoding="utf-8-sig")
        log_download("SUCCESS", tag, f"shape={df.shape}")
        return df
    except Exception as e:
        log_download("FAILED", tag, f"Error: {e}")
        return pd.DataFrame()

def fetch_stock_finance(code):
    raw = ak.stock_financial_abstract(symbol=code)
    if raw is None or raw.empty:
        return pd.DataFrame()
    rows = []
    for _, row in raw.iterrows():
        indicator = str(row["指标"])
        if indicator not in FINANCE_INDICATORS:
            continue
        for col in raw.columns[2:]:
            period = str(col)
            if not period[:4].isdigit() or not period.endswith("1231"):
                continue
            val = row[col]
            if pd.isna(val) or val == "--":
                continue
            rows.append({
                "code": str(code).zfill(6),
                "year": int(period[:4]),
                "indicator": indicator,
                "value": float(val),
            })
    return pd.DataFrame(rows)

def download_all_finance():
    tag = "finance_ratios"
    frames = []
    for item in STOCK_LIST:
        code = item["code"]
        try:
            df = fetch_stock_finance(code)
            if not df.empty:
                frames.append(df)
            log_download("SUCCESS", f"finance_{code}", f"rows={len(df)}")
            time.sleep(0.5)
        except Exception as e:
            log_download("FAILED", f"finance_{code}", f"Error: {e}")
    if not frames:
        log_download("FAILED", tag, "Error: no finance data")
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out = out[out["year"] >= out["year"].max() - 4]
    out.to_csv(DIRS["finance"] / "finance_ratios.csv", index=False, encoding="utf-8-sig")
    log_download("SUCCESS", tag, f"shape={out.shape}")
    return out

print("下载函数已定义")

下载函数已定义


## 1.1 股票日度行情（后复权）

下载 10 只股票 2020-01-01 至今的日度后复权数据，字段包含：日期、开盘价、收盘价、最高价、最低价、成交量、成交额。

In [3]:
stocks = download_all_stocks()
print(f'成功下载 {len(stocks)} 只股票')
if stocks:
    display(list(stocks.values())[0].head())

成功下载 10 只股票


,date,open,close,high,low,volume,amount,code
0,2020-01-02,8.73,8.78,8.85,8.72,2349494,1.404443e+09,601398
1,2020-01-03,8.78,8.81,8.84,8.77,1522130,9.119516e+08,601398
2,2020-01-06,8.77,8.78,8.87,8.76,2265097,1.359917e+09,601398
3,2020-01-07,8.80,8.83,8.86,8.80,1168044,7.016158e+08,601398
4,2020-01-08,8.77,8.72,8.78,8.71,1585591,9.405527e+08,601398


## 1.2 市场指数

下载沪深 300和创业板指。

In [4]:
indices = download_all_indices()
print(f'成功下载 {len(indices)} 个指数')

成功下载 2 个指数


## 1.3 宏观经济指标

- CPI 同比增速：衡量通货膨胀水平，直接影响货币政策预期和实际利率。
- 人民币/美元汇率：影响外资流向和进出口企业盈利，对 A 股估值有系统性影响。

In [5]:
cpi_df = download_macro_cpi()
m2_df = download_macro_m2()
print('CPI', cpi_df.shape, 'M2', m2_df.shape)

CPI (68, 3) M2 (76, 3)


## 1.4 财务指标（长格式）

获取最近 5 个年度的净资产收益率（ROE）和销售毛利率。

In [6]:
finance_df = download_all_finance()
if finance_df.empty:
    print('财务数据为空，请检查网络')
else:
    display(finance_df.head(10))

,code,year,indicator,value
0,601398,2025,净资产收益率(ROE),9.450000
1,601398,2024,净资产收益率(ROE),9.880000
2,601398,2023,净资产收益率(ROE),10.660000
3,601398,2022,净资产收益率(ROE),11.450000
4,601398,2021,净资产收益率(ROE),12.150000
17,601398,2025,销售净利率,44.229902
18,601398,2024,销售净利率,44.651333
19,601398,2023,销售净利率,43.307910
20,601398,2022,销售净利率,39.329229
21,601398,2021,销售净利率,37.147869


## 1.5 下载日志

以上所有下载结果已记录到 `download_log.txt`。

In [7]:
print(LOG_FILE.read_text(encoding='utf-8') if LOG_FILE.exists() else '无日志')

[2026-05-19 10:08:38] SUCCESS stock_601398  shape=(1542, 8)
[2026-05-19 10:08:39] SUCCESS stock_600036  shape=(1542, 8)
[2026-05-19 10:08:40] SUCCESS stock_002594  shape=(1542, 8)
[2026-05-19 10:08:41] SUCCESS stock_601633  shape=(1542, 8)
[2026-05-19 10:08:41] SUCCESS stock_000002  shape=(1542, 8)
[2026-05-19 10:08:42] SUCCESS stock_600519  shape=(1542, 8)
[2026-05-19 10:08:43] SUCCESS stock_000858  shape=(1542, 8)
[2026-05-19 10:08:44] SUCCESS stock_601857  shape=(1542, 8)
[2026-05-19 10:08:44] SUCCESS stock_600941  shape=(1055, 8)
[2026-05-19 10:08:45] SUCCESS stock_002352  shape=(1539, 8)
[2026-05-19 10:08:46] SUCCESS index_000300  shape=(1541, 7)
[2026-05-19 10:08:47] SUCCESS index_399006  shape=(1541, 7)
[2026-05-19 10:08:55] SUCCESS macro_cpi  shape=(68, 3)
[2026-05-19 10:08:55] SUCCESS macro_m2  shape=(76, 3)
[2026-05-19 10:08:57] SUCCESS finance_601398  rows=90
[2026-05-19 10:08:58] SUCCESS finance_600036  rows=108
[2026-05-19 10:08:59] SUCCESS finance_002594  rows=84
[2026-05